# Lymphoid Functional Scores, Composition and Checkpoint Markers

Scores T-cell functional modules (naïve, activation, exhaustion, cytotoxicity, …) on
the annotated lymphoid object from 1.3a, summarises them by cell state and by
cell state × treatment stage, plots per-patient state composition, and
profiles checkpoint / co-stimulatory markers in key T-cell states.

In [ ]:
# !pip install --quiet scanpy
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/lymphoid

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.cluster.hierarchy import leaves_list, linkage

## Config

In [ ]:
input_path = "sub_Lymphoid_raw_scvi_combine_annotated.h5ad"

signature_path = "Tcell_score.csv"            # one module per column, genes down the rows
figures_output_dir = "./figures"
tables_output_dir = "./tables"

counts_layer = "raw_counts"
state_col = "cell_state"
recist_col = "recist_v3"
sample_col = "ID_2"
seed = 0

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.set_figure_params(dpi=80, figsize=(4, 4))

## 1. Load and normalise

In [ ]:
adata = sc.read(input_path)
adata.X = adata.layers[counts_layer].copy() if counts_layer in adata.layers else adata.X.copy()
adata.uns.pop("log1p", None)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

state_order = [
    "Plasma cell", "B cell", "B cell dividing", "CD4 T naive", "CD4 Tem", "CD4 T regulatory",
    "CD8 T naive", "CD8 Tem", "CD8 Tem_ENTPD1high", "NK cell", "Cytotoxic NK cell",
]

missing_states = sorted(set(state_order) - set(adata.obs[state_col].astype(str)))
assert not missing_states, f"states in state_order not in data: {missing_states}"
print(adata.n_obs, "cells")
pd.crosstab(adata.obs[state_col], adata.obs[recist_col])

## 2. Score T-cell modules

Module names are cleaned (`:` `/` and spaces → `_`). Modules with fewer than
`min_module_genes` genes in the data are skipped; the control set size equals the
number of genes present.

In [ ]:
# --- module scoring (sc.tl.score_genes) ---
min_module_genes = 5
score_n_bins = 50
plot_modules = [                               # modules shown in heatmaps, in this order
    "Naïve", "Activation_Effector_function", "Exhaustion", "TCR_Signaling", "Cytotoxicity",
    "Cytokine_Cytokine_receptor", "Chemokine_Chemokine_receptor", "Senescence", "Anergy",
    "IFN_Response", "NFKB_Signaling", "Adhesion",
]
umap_modules = ["Exhaustion", "Cytotoxicity", "Naïve", "Activation_Effector_function"]
gep_genes = ["HLA-DQA1", "PSMB10", "HLA-DRB1", "HLA-E", "CD276", "CD38", "CXCL9", "LAG3", "IDO1", "STAT1", "ENTPD1",
             "PDCD1LG2", "CD274", "CMKLR1", "CXCR6", "CCL5", "CD8A", "NKG7", "TIGIT", "CD27", "CTLA4"]


def clean_name(name):
    return name.replace(":", "_").replace("/", "_").replace(" ", "_")

gene_df = pd.read_csv(signature_path)
gene_df = gene_df.loc[:, gene_df.notna().any()]
modules = {clean_name(c): gene_df[c].dropna().astype(str).str.strip().tolist() for c in gene_df.columns}

coverage = []
for module, genes in modules.items():
    present = [g for g in dict.fromkeys(genes) if g in adata.var_names]
    coverage.append({"module": module, "n_genes": len(set(genes)), "n_present": len(present),
                     "missing": ", ".join(sorted(set(genes) - set(present)))})
    if len(present) >= min_module_genes:
        sc.tl.score_genes(adata, gene_list=present, ctrl_size=len(present), n_bins=score_n_bins,
                          score_name=f"{module}_score", random_state=seed)

present_gep = [g for g in gep_genes if g in adata.var_names]
sc.tl.score_genes(adata, gene_list=present_gep, ctrl_size=len(present_gep), n_bins=score_n_bins,
                  score_name="GEP_score", random_state=seed)
coverage.append({"module": "GEP", "n_genes": len(gep_genes), "n_present": len(present_gep),
                 "missing": ", ".join(sorted(set(gep_genes) - set(present_gep)))})

coverage_df = pd.DataFrame(coverage)
coverage_df.to_csv(f"{tables_output_dir}/lymphoid_module_gene_coverage.csv", index=False)
score_keys = [f"{m}_score" for m in plot_modules if f"{m}_score" in adata.obs]
print("not scored / not found:", [m for m in plot_modules if f"{m}_score" not in adata.obs])
coverage_df

## 3. Module scores by cell state

Matrixplot (min-max scaled per module) and a z-scored heatmap of mean score per state
(states with ≥ `min_cells_per_group` cells). `*` marks z > `star_z_state`.

In [ ]:
# --- score heatmaps (mean score per group, z-scored across groups) ---
min_cells_per_group = 50
star_z_state = 0.5                             # "*" where z > this (display threshold, not a test)
star_z_state_stage = 0.25
stage_exclude_ids = ["R008", "R011"]           # liver biopsies excluded from state x stage summaries

def zscored_means(obs, group_col, order):
    sizes = obs[group_col].value_counts()
    keep = sizes[sizes >= min_cells_per_group].index
    means = obs[obs[group_col].isin(keep)].groupby(group_col, observed=True)[score_keys].mean()
    means = means.loc[[g for g in order if g in means.index]]
    return (means - means.mean()) / means.std(), sizes

def star_heatmap(z, threshold, figsize, save_path=None, title=None, **kwargs):
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(z, cmap="coolwarm", center=0, cbar_kws={"label": "Z-score"}, ax=ax, **kwargs)
    for i, j in zip(*np.where(z.values > threshold)):
        ax.text(j + 0.5, i + 0.5, "*", ha="center", va="center", color="black", fontsize=10)
    if title:
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    return fig, ax

sc.pl.matrixplot(adata, var_names=score_keys, groupby=state_col, standard_scale="var", cmap="coolwarm",
                 dendrogram=True, swap_axes=True, figsize=(5, 4), save="_overall.pdf")

state_z, _ = zscored_means(adata.obs, state_col, state_order)
state_z.to_csv(f"{tables_output_dir}/lymphoid_module_scores_by_state_z.csv")
star_heatmap(state_z.T, star_z_state, (6, 4.5), save_path=f"{figures_output_dir}/matrixplot_overall_with_stars.pdf")

## 4. Module scores by cell state × treatment stage

Liver biopsies (`stage_exclude_ids`) are excluded so TKI-resistant groups contain lung
tissue only. Groups with < `min_cells_per_group` cells are dropped. Columns are grouped
by cell state (blocks), stages ordered within each block; modules are clustered.
`*` marks z > `star_z_state_stage`.

In [ ]:
obs = adata.obs[~adata.obs["ID"].isin(stage_exclude_ids)].copy()
obs["recist_cellstate"] = obs[recist_col].astype(str) + "-" + obs[state_col].astype(str)

recist_order = ["0_adj_normal", "1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]


group_order = [f"{r}-{s}" for s in state_order for r in recist_order]
stage_z, sizes = zscored_means(obs, "recist_cellstate", group_order)
stage_z.to_csv(f"{tables_output_dir}/lymphoid_module_scores_by_state_stage_z.csv")
print("dropped groups (< min cells):", sorted(set(sizes.index) - set(stage_z.index)))

matrix = stage_z.T
matrix = matrix.iloc[leaves_list(linkage(matrix.values, method="ward"))]
col_states = [c.split("-", 1)[1] for c in matrix.columns]
boundaries = [i for i in range(1, len(col_states)) if col_states[i] != col_states[i - 1]] + [len(col_states)]

fig, ax = star_heatmap(matrix, star_z_state_stage, (10, 8), xticklabels=True, yticklabels=True)
for b in boundaries[:-1]:
    ax.axvline(b, color="black", lw=1.5)
start = 0
for b in boundaries:
    ax.text((start + b) / 2, -0.8, col_states[start], ha="center", va="bottom", fontsize=9, fontweight="bold", rotation=45)
    start = b
ax.set(xlabel="Cell state blocks → RECIST stage within each block", ylabel="Modules (clustered)")
fig.savefig(f"{figures_output_dir}/matrixplot_recist_cellstate_row_clustered_blocked.pdf", dpi=300, bbox_inches="tight")

### Per-patient mean scores (robustness)

In [ ]:
per_patient = (obs.groupby([state_col, recist_col, sample_col], observed=True)
               .agg(n_cells=(score_keys[0], "size"), **{k: (k, "mean") for k in score_keys})
               .reset_index())
per_patient = per_patient[per_patient["n_cells"] >= min_cells_per_sample]
per_patient.to_csv(f"{tables_output_dir}/lymphoid_module_scores_per_patient.csv", index=False)
per_patient.groupby([state_col, recist_col], observed=True).size().unstack(fill_value=0).reindex(state_order)

## 5. Score UMAPs

In [ ]:
sc.pl.umap(adata, color=[f"{m}_score" for m in umap_modules if f"{m}_score" in adata.obs],
           use_raw=False, color_map="OrRd", size=100, save="_exhautive_cytotoxic_score.pdf")
sc.pl.umap(adata, color="GEP_score", use_raw=False, color_map="OrRd", size=100)

## 6. State composition per patient 

% of each state among all lymphoid cells in the patient. Patients with fewer than
`min_cells_per_sample` cells are greyed out; the top bar gives n.

In [ ]:
# --- per-patient composition (Sup. Fig. 6) ---
composition_order = [                          # low-count samples (TN008, P002, P003, P008, R003, R006, R012) omitted
    "TN009_AN", "TN007_AN", "TN005_AN", "TN003_AN", "TN001_AN", "TN002_AN", "TN004_AN", "TN008_AN", "TN006_AN",
    "TN004", "TN001", "TN002", "TN005", "TN006", "TN009", "TN007", "TN003",
    "P013", "P006", "P005", "P012", "P001", "P011", "P009",
    "E001", "O001",
    "D001", "D002", "D003",
    "R009", "R007", "R002", "R004", "R001", "R010", "R005",
    "R008", "R011",
]
min_cells_per_sample = 20                      # samples below this are greyed out

counts = pd.crosstab(adata.obs[sample_col].astype(str), adata.obs[state_col].astype(str))
counts = counts.reindex([s for s in composition_order if s in counts.index])
counts = counts.reindex(columns=state_order, fill_value=0)
total = counts.sum(axis=1)
percent = counts.div(total.replace(0, np.nan), axis=0) * 100
low = total < min_cells_per_sample
pd.concat([counts, total.rename("n_cells"), percent.add_suffix(" (%)")], axis=1).to_csv(
    f"{tables_output_dir}/lymphoid_state_percent_per_patient.csv")

fig, axes = plt.subplots(2, 2, figsize=(10, 4), gridspec_kw={
    "height_ratios": [0.6, 3], "width_ratios": [40, 1], "hspace": 0.05, "wspace": 0.02})
ax_n, ax, cax = axes[0, 0], axes[1, 0], axes[1, 1]
axes[0, 1].axis("off")
ax_n.bar(np.arange(len(total)) + 0.5, total.values, color=np.where(low, "lightgrey", "black"), width=0.8)
ax_n.set_xlim(0, len(total)); ax_n.set_xticks([]); ax_n.set_yticks([0, int(total.max())])
ax_n.set_ylabel("n cells", fontsize=8)
for side in ("top", "right"):
    ax_n.spines[side].set_visible(False)
sns.heatmap(percent.mask(low, axis=0).T, cmap="mako_r", linewidths=0.25, vmin=0, vmax=100,
            cbar_ax=cax, cbar_kws={"label": "% of lymphoid cells per patient"}, ax=ax)
for j in np.where(low)[0]:
    ax.add_patch(plt.Rectangle((j, 0), 1, len(state_order), color="lightgrey", zorder=0))
ax.set(xlabel="Patient ID", ylabel="Cell state")
ax.tick_params(axis="x", rotation=90)
plt.savefig(f"{figures_output_dir}/all_cell_type_rep_prop_by_subset_patients.pdf", bbox_inches="tight")
plt.show()
print(f"greyed (< {min_cells_per_sample} cells):", list(total.index[low]))

## 7. Checkpoint and co-stimulatory markers in key T-cell states

Primary-lung samples with adequate cell numbers only (`checkpoint_exclude_ids`);
state × stage groups with ≤ `min_cells_checkpoint_group` cells are dropped. The
cell-count bar shares the dotplot's group order.

In [ ]:

# --- checkpoint dotplot in key T-cell states (primary lung, adequate samples) ---
checkpoint_exclude_ids = ["R008", "R011", "R012", "R003", "R006", "TN008", "P008", "P002", "P003"]
checkpoint_states = ["CD4 T regulatory", "CD8 T naive", "CD8 Tem", "CD8 Tem_ENTPD1high"]
min_cells_checkpoint_group = 20
checkpoint_panel = {
    "FDA_Checkpoint_targets": ["CD274", "PDCD1", "CTLA4", "LAG3", "TIGIT", "HAVCR2", "VSIR"],
    "Co_stimulatory_receptors": ["CD28", "ICOS", "CD27", "TNFRSF4", "TNFRSF9", "TNFRSF18"],
    "Terminal_dysfunction_targets": ["LAYN"],
    "Adhesion_synapse": ["ITGAE", "ITGAL", "CD2"],
    "HLA": ["HLA-A", "HLA-B", "HLA-C"],
    "chemokine1": ["CCL5", "AOAH", "CCL4", "IFNG"],
    "chemokine2": ["CXCL13", "IL21R"],
}


t = adata[~adata.obs["ID"].isin(checkpoint_exclude_ids) & adata.obs[state_col].isin(checkpoint_states)].copy()
t.obs["cellstate_recist"] = t.obs[state_col].astype(str) + "_" + t.obs[recist_col].astype(str)
sizes = t.obs["cellstate_recist"].value_counts()
order = [f"{s}_{r}" for s in checkpoint_states for r in recist_order
         if sizes.get(f"{s}_{r}", 0) > min_cells_checkpoint_group]
t = t[t.obs["cellstate_recist"].isin(order)].copy()
t.obs["cellstate_recist"] = pd.Categorical(t.obs["cellstate_recist"], categories=order)

panel = {k: [g for g in v if g in t.var_names] for k, v in checkpoint_panel.items()}
missing = {k: sorted(set(v) - set(panel[k])) for k, v in checkpoint_panel.items() if set(v) - set(panel[k])}
if missing:
    print("not in data:", missing)
axes_dict = sc.pl.dotplot(t, {k: v for k, v in panel.items() if v}, groupby="cellstate_recist", use_raw=False,
                          standard_scale="var", cmap="Blues", figsize=(7.5, 4.5), show=False)
for ax in axes_dict.values():
    for label in ax.get_xticklabels():
        label.set_rotation(90)
        label.set_style("italic")
plt.savefig(f"{figures_output_dir}/dotplot_Tcells_cellstate_key_markers.pdf", bbox_inches="tight")
plt.show()

n = t.obs["cellstate_recist"].value_counts().reindex(order)
fig, ax = plt.subplots(figsize=(2, 4.5))
ax.barh(np.arange(len(n)), n.values, color="steelblue", alpha=0.7, height=0.8)
ax.set_yticks(np.arange(len(n)), n.index, fontsize=9)
ax.invert_yaxis()
ax.set(xlabel="Cell count", title="N cells")
for i, v in enumerate(n.values):
    ax.text(v + n.max() * 0.02, i, str(v), va="center", fontsize=8)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.savefig(f"{figures_output_dir}/cellcounts_Tcells_cellstate.pdf", bbox_inches="tight", dpi=300)
plt.show()